# Black-Box Optimisation (BBO) Capstone Project
## 1. Imports

In [ ]:
# Numerical & scientific
import numpy as np
import pandas as pd
import itertools
from scipy.stats import norm
from scipy.spatial.distance import cdist

# Plotting
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import math

# Machine Learning / GP
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic, Matern, ConstantKernel, WhiteKernel
from sklearn.cluster import DBSCAN

# Experimental design
from pyDOE2 import lhs

## 2. Data Loading and Aggregation

In [ ]:
# Choose function
function=1

In [ ]:
# Load inital data
input_path = f"initial_data/function_{function}/initial_inputs.npy"
output_path = f"initial_data/function_{function}/initial_outputs.npy"

X = np.load(input_path)
y = np.load(output_path)

#Load weekly X submitted and y received
input_path_X_weeks = f"weekly_data/X_weeks.npy"
input_path_y_weeks = f"weekly_data/y_weeks.npy"

X_weeks = np.load(input_path_X_weeks, allow_pickle=True)
y_weeks = np.load(input_path_y_weeks, allow_pickle=True)

for week_X, week_y in zip(X_weeks, y_weeks):
    X = np.vstack([X, week_X[function-1].reshape(1, X.shape[1])])
    y = np.append(y, week_y[function-1])

# Check the shape and contents

print(f"Shape(X): {X.shape}")
print(f"Mean(X): {round(X.mean(),4)}")
print(f"Std(X): {round(np.std(X),4)}")
#print("X:\n", X)
print()

print(f"Shape(y): {y.shape}")
print(f"Mean(y): {round(y.mean(),4)}")
print(f"Std(y): {round(y.std(), 4)}")
#print("y:\n", y)
print()

y_std =(y - y.mean()) / y.std()
#print("y standardized:\n", y_std)

print(f"Current max(y): {round(y.max(), 4)}, at X: {X[y.argmax()]}")

## 3. Exploratory Data Analysis and Visual Diagnostics

In [ ]:
#Check correlation

corr_df = pd.DataFrame(
    np.corrcoef(X, rowvar=False),
    index=[f"x{i+1}" for i in range(X.shape[1])],
    columns=[f"x{i+1}" for i in range(X.shape[1])]
)

corr_df.style.format("{:.4f}").background_gradient(
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

In [ ]:
#Plot y (Blue: initial coordinates, red: predicted coordinates)

split_idx = len(y)-len(y_weeks)

#Plot y
plt.figure(figsize=(8,5))
x = np.arange(1, len(y) + 1)
plt.bar(x[:split_idx], y[:split_idx], color='lightblue', label='Initial')
plt.bar(x[split_idx:], y[split_idx:], color='salmon', label='Predicted')
plt.xlabel('Index')
plt.ylabel('y value')
plt.title(f"Function {function}: y (Initial vs Predicted)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

#Plot y_std
plt.figure(figsize=(8,5))
x = np.arange(1, len(y) + 1)
plt.bar(x[:split_idx], y_std[:split_idx], color='lightblue', label='Initial')
plt.bar(x[split_idx:], y_std[split_idx:], color='salmon', label='Predicted')
plt.xlabel('Index')
plt.ylabel('y value')
plt.title(f"Function {function}: y (Initial vs Predicted)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend()
plt.show()

#Plot addition graph for functions in 3D
if X.shape[1]<3:
    fig = plt.figure(figsize=(16,10))
    ax = fig.add_subplot(111, projection='3d')
    dx = dy = 0.03
    ax.bar3d(X[:,0], X[:,1], np.zeros_like(y), dx, dy, y, color='blue', alpha=0.8)
    ax.set_xlim(0, 1)  # give a little space above/below tallest/shortest bar
    ax.set_ylim(0, 1)  # give a little space above/below tallest/shortest bar
    ax.set_zlim(np.min(y)-0.1*abs(np.min(y)), np.max(y)+0.1*abs(np.max(y)))  # give a little space above/below tallest/shortest bar
    plt.title(f"3D Space Visualization (Function {function})")

In [ ]:
#Plot X coordinates in all dimensions (Blue: initial coordinates, red: predicted coordinates)

def auto_subplot_grid(n_plots, ncols=3, figsize_per_plot=5):
    """
    Creates an adaptive subplot grid for any number of plots.
    """
    nrows = math.ceil(n_plots / ncols)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_plot * ncols, figsize_per_plot * nrows)
    )

    axes = axes.ravel() if n_plots > 1 else [axes]

    # Hide unused axes
    for i in range(n_plots, len(axes)):
        axes[i].set_visible(False)

    return fig, axes


def plot_all_pairs(X, y , highlight_last_n=1, ncols=3):
    """
    Pairwise scatter plot function.

    X: (n_samples, n_dims)
    y: optional (not used for now)
    highlight_last_n: number of last points to highlight in red
    ncols: number of columns in grid
    """
    n_dims = X.shape[1]
    pairs = list(itertools.combinations(range(n_dims), 2))

    fig, axes = auto_subplot_grid(len(pairs), ncols=ncols)

    for idx, (i, j) in enumerate(pairs):
        ax = axes[idx]

        # Base points (blue)
        ax.scatter(X[:, i], X[:, j], color='blue', s=70)

        # Highlight last N points (red)
        ax.scatter(
            X[-highlight_last_n:, i],
            X[-highlight_last_n:, j],
            color='red',
            s=100
        )

        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.set_xlabel(f"X{i+1}")
        ax.set_ylabel(f"X{j+1}")
        ax.set_title(f"Dimensions {i+1} vs {j+1}")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_all_pairs(X, y, highlight_last_n= len(X_weeks))

## 4. Maximum-Minimum Distance Exploration (LHS-Based)

In [ ]:
# Maximises exploration of under-sampled regions:
# Generate candidate points using Latin Hypercube Sampling
# Select the point maximising distance to existing observations

n_candidates = 200000 #set higher for more precise result
candidates = lhs(X.shape[1], samples=n_candidates)  # values in [0,1]

# Compute distances from each candidate to all existing points
distances = cdist(candidates, X)

# For each candidate, distance to nearest existing point
min_dist = distances.min(axis=1)

# Index of the most unexplored candidate
best_idx = np.argmax(min_dist)

# The point to sample next
unexp_coordinate = candidates[best_idx]
print(unexp_coordinate.round(6))

## 5. Gaussian Process Kernel Selection (Marginal Likelihood Analysis)

In [ ]:
# Compare marginal likelihoods across Matérn kernels (ν = 0.5, 1.5, 2.5) and an RBF kernel.
# Length-scale and signal-variance bounds are constrained to sanity-check the optimization.
# Note: the optimal ν is selected based on global (domain-wide) model fit, not local behavior.

my_nu=[0.5, 1.5, 2.5]
l_bounds= (0.01, 0.5)
std_bounds=(0.1, 5)

for i in my_nu:
    kernel = ConstantKernel(1.0, std_bounds) * Matern(length_scale=0.1, length_scale_bounds= l_bounds, nu=i)
    gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
    gpr.fit(X, y)
    LML = gpr.log_marginal_likelihood()
    print(f"MLL nu=({i}) = {LML:.4f}, kernel optimized: {gpr.kernel_}")

#RBF option
kernel = ConstantKernel(1.0, std_bounds) * RBF(length_scale=0.1, length_scale_bounds=l_bounds)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gpr.fit(X, y)
    
LML = gpr.log_marginal_likelihood()
print(f"MLL(RBF) = {LML:.4f}, kernel optimized: {gpr.kernel_}")

## 6. Local Curvature-Based Selection of Matérn Smoothness Parameter (ν)

In [ ]:
def select_nu(X, y, k=5):
    
    """
    Estimate local curvature around the current maximum using k-nearest neighbours.

    This method provides a geometry-based heuristic for selecting the Matérn smoothness
    parameter ν by analysing local variation in the objective function rather than relying
    on global marginal likelihood maximisation.

    Local curvature is quantified using normalised slopes between the current maximum
    and its nearest k neighbours in the input space. This allows distinguishing between:

        - Sharp, high-curvature regions → favour lower ν (rougher kernels)
        - Moderate curvature regions → favour higher ν (smoother kernels)

    Parameters
    X : ndarray, shape (n_samples, n_features)
        Input locations in the search space.
    y : ndarray, shape (n_samples,)
        Observed objective values (assumed standardised externally).
    k : int, default=5
        Number of nearest neighbours used for local analysis.

    Returns
    slopes : ndarray
        Local slope estimates used for curvature assessment.
    """

    # find maxima
    i_max = np.argmax(y_std)
    x_max = X[i_max]
    y_max = y_std[i_max]
    
    # compute distances
    dists = np.linalg.norm(X - x_max, axis=1)
    
    # get k nearest neighbors (exclude itself)
    idx_sorted = np.argsort(dists)
    idx = idx_sorted[1:k+1]
    
    # compute slopes
    slopes = np.abs(y_max - y[idx]) / ((dists[idx]) / np.sqrt(len(X)))
    
    # print table with normalised distances
    print(f"y_max:  {y_max:.4f}")
    print(f"x_max:  {x_max}")
    #print(f"max distance:  {np.sqrt(len(x_max)):.4f}")
    #print(f"typical distance:  {np.sqrt(len(x_max)/6):.4f}\n")
    print(f"max distance norm.: 1.0000") # given by sqrt(D), normalized as sqrt(D/6) / sqrt(D)
    print(f"typical distance norm.: 0.4082") #given by sqrt(D/6), normalized as sqrt(D/6) / sqrt(D)
    
    print("")
    
    print(f"{'obj. value (y) ':^15} | {'distance norm.':^15} | {'slope':^15}")
    print("-" * 50)
    
    for j, s in zip(idx, slopes):
        print(f"{y[j]:^15.4f} | {dists[j]:^15.4f} | {s:^15.4f}")

select_nu(X, y, k=5)

## 7. Gaussian Process Kernel Selection and Configuration

In [ ]:
#Choose Kernel (Matern of RBF) and Length Scale Bounds based on previous observations
kernel= "Matern" #choose "Matern" or "RBF"
my_nu = 2.5
l_bounds= (0.01, 0.5)
std_bounds=(0.1, 5)

# Corrected kernel assignment
if kernel == "Matern":
    kernel = ConstantKernel(1.0, std_bounds) * Matern(length_scale=0.1, length_scale_bounds=l_bounds, nu=my_nu)
elif kernel == "RBF":
    kernel = ConstantKernel(1.0, std_bounds) * RBF(length_scale=0.1, length_scale_bounds=l_bounds)

#Fit Gaussian Process Regressor
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, normalize_y=True)
gpr.fit(X, y)
print(f"Kernel optimized: {gpr.kernel_}")

## 8. Candidate Point Generation for Acquisition

In [ ]:
#Generate Candidate Points

if len(X[0])<3:
    # 2D grid for low-dimensional inputs (2D)
    x1 = np.linspace(0, 1, 1000)
    x2 = np.linspace(0, 1, 1000)
    X1, X2 = np.meshgrid(x1, x2)
    candidates = np.column_stack([X1.ravel(), X2.ravel()])
else:
    # High-dimensional inputs (3D+): LHS
    n_candidates = 2000000
    candidates = lhs(X.shape[1], samples=n_candidates)  # values in [0,1]

## 9. Expected Improvement Acquisition Function

In [ ]:
#Set hyperparameter xi to balance exploration/exploitation
my_xi=0.01

In [ ]:
#Define Prediction Function with Expected Improvement
def select_next_point_ei(candidates, y, xi, show_kernel= True):
    
    """
    Select the next evaluation point by maximising the Expected Improvement (EI)
    acquisition function over a set of candidate inputs.

    For each candidate point, the Gaussian Process surrogate is used to predict
    the posterior mean and standard deviation. Expected Improvement is then
    computed relative to the best observed objective value so far, incorporating
    an exploration parameter xi to control the exploration–exploitation trade-off.
    The candidate with the highest EI is selected as the next query point.

    Parameters
    candidates : ndarray of shape (n_candidates, n_features)
        Candidate input locations at which Expected Improvement is evaluated.
        These are typically generated locally around the current best point.
    y : ndarray of shape (n_samples,)
        Observed objective values from previously evaluated points.
    xi : float
        Exploration parameter controlling the trade-off between exploration
        and exploitation in the Expected Improvement criterion.
    show_kernel : bool, default=True
        If True, prints the optimised Gaussian Process kernel after prediction.

    Returns
    next_point : ndarray of shape (n_features,)
        Candidate point with the highest Expected Improvement, selected as the
        next input to evaluate.
    """

    y_mean, y_std = gpr.predict(candidates, return_std=True)
    
    ## best observed value
    f_best = np.max(y)
    xi = my_xi    
    
    mu = y_mean # rename for clarity
    sigma = y_std
    
    sigma_safe = np.maximum(sigma, 1e-12) # avoid division by zero
    
    Z = (mu - f_best - xi) / sigma_safe
  
    EI = (mu - f_best - xi) * norm.cdf(Z) + sigma_safe * norm.pdf(Z)
    EI[sigma < 1e-12] = 0.0   # define improvement = 0 if no uncertainty
    
    # Select next candidate with highest EI
    max_idx = np.argmax(EI)
    
    next_point = candidates[max_idx]
    
    print(f"New prediction: {'-'.join(f'{v:.6f}' for v in next_point)}")
    if show_kernel == True:
        print(f"kernel optimized: {gpr.kernel_}")
    return next_point

#Predict Next Candidate Point
x_pred=select_next_point_ei(candidates, y, my_xi)

## 10. Hierarchical Local Refinement Around Current Optimum

In [ ]:
#Zoom-In Search Around Current Best Point

my_points_per_dim = 10  # generated points to inspect, lower for higher dimensions
zoom_levels = np.arange(7)  # number of "zooms", set equal to the 6 decimal places
repeats_per_zoom = 10       # local attempts per zoom, higher for higher dimensions

# Assume x_pred is already defined as your current best point
for level in zoom_levels:
    delta = 0.1 / (10 ** level)
    print(f"Zoom {level+1}  (±1e-0{level+1})")
    
    for _ in range(repeats_per_zoom):
        n_dims = len(x_pred)
        
        # Create grids for each dimension
        grids = [
            np.linspace(max(0.0, x_pred[i]-delta), min(1.0, x_pred[i]+delta), my_points_per_dim)
            for i in range(n_dims)
        ]
        
        # Generate candidate points
        if n_dims == 2:
            # Use meshgrid for 2D (faster)
            X1, X2 = np.meshgrid(grids[0], grids[1])
            new_candidates = np.column_stack([X1.ravel(), X2.ravel()])
        else:
            # Use Cartesian product for higher dimensions
            new_candidates = np.array(list(itertools.product(*grids)))
        
        # Update best point
        x_pred = select_next_point_ei(new_candidates, y, my_xi, show_kernel=False)  # returns best point

## 11. Evaluation History and Final Predicted and Current Optimum

In [ ]:
#Print Observed X Values in Previous Rounds
X_for_function = [week[function - 1] for week in X_weeks]

print(f"### Function: {function} ###")
for i, x in enumerate(X_for_function, start=1):
    formatted = "-".join(f"{v:.6f}" for v in x)
    print(f"Week {i}: {formatted}")
print("")

print(f"Final prediction function {function}: {'-'.join(f'{v:.6f}' for v in x_pred)}")